# Landing prediction model comparison

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Reproducible holdout and course baseline
Use 72 training and 18 test records with random_state=2. For comparability, the historical course baseline scales before the split. The second analysis corrects this leakage with a pipeline that fits scaling inside each training fold. Both are labeled explicitly. A tree random_state=2 removes nondeterministic results.

In [2]:
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,balanced_accuracy_score
X=pd.read_csv(OUT/'dataset_part_3.csv');y=pd.read_csv(OUT/'dataset_part_2.csv').Class.to_numpy()
train_idx,test_idx=train_test_split(np.arange(len(y)),test_size=.2,random_state=2)
print('Training:',len(train_idx),'Test:',len(test_idx))
models={
'Logistic regression':(LogisticRegression(max_iter=3000),{'C':[.01,.1,1],'solver':['lbfgs']}),
'SVM':(SVC(),{'kernel':['linear','rbf','poly','sigmoid'],'C':np.logspace(-3,3,5),'gamma':np.logspace(-3,3,5)}),
'Decision tree':(DecisionTreeClassifier(random_state=2),{'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':list(range(2,20,2)),'max_features':['sqrt'],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}),
'KNN':(KNeighborsClassifier(),{'n_neighbors':list(range(1,11)),'algorithm':['auto'],'p':[1,2]})}
# Modern sklearn removed max_features='auto' for trees. Its historical classifier meaning was 'sqrt'.
scaled=StandardScaler().fit_transform(X)
results=[];fitted={}
for name,(model,params) in models.items():
    search=GridSearchCV(model,params,cv=10,n_jobs=2,scoring='accuracy')
    search.fit(scaled[train_idx],y[train_idx]);pred=search.predict(scaled[test_idx])
    results.append({'Protocol':'Course baseline','Model':name,'CV':search.best_score_,'Test':accuracy_score(y[test_idx],pred),'BalancedAccuracy':balanced_accuracy_score(y[test_idx],pred),'Params':search.best_params_,'ConfusionMatrix':confusion_matrix(y[test_idx],pred).tolist()})
    print(name,results[-1])


Training: 72 Test: 18


Logistic regression {'Protocol': 'Course baseline', 'Model': 'Logistic regression', 'CV': np.float64(0.8464285714285713), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'C': 0.01, 'solver': 'lbfgs'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


SVM {'Protocol': 'Course baseline', 'Model': 'SVM', 'CV': np.float64(0.8482142857142856), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'C': np.float64(1.0), 'gamma': np.float64(0.03162277660168379), 'kernel': 'sigmoid'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


Decision tree {'Protocol': 'Course baseline', 'Model': 'Decision tree', 'CV': np.float64(0.8767857142857143), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'criterion': 'gini', 'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'splitter': 'best'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}
KNN {'Protocol': 'Course baseline', 'Model': 'KNN', 'CV': np.float64(0.8482142857142858), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'algorithm': 'auto', 'n_neighbors': 10, 'p': 1}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


## Training-only preprocessing
Each cross-validation fold fits its own StandardScaler. Test labels are never used to select hyperparameters. Model selection uses training cross-validation accuracy.

In [3]:
for name,(model,params) in models.items():
    pipeline=Pipeline([('scale',StandardScaler()),('model',model)])
    grid={'model__'+k:v for k,v in params.items()}
    search=GridSearchCV(pipeline,grid,cv=10,n_jobs=2,scoring='accuracy')
    search.fit(X.iloc[train_idx],y[train_idx]);pred=search.predict(X.iloc[test_idx])
    results.append({'Protocol':'Train-only scaling','Model':name,'CV':search.best_score_,'Test':accuracy_score(y[test_idx],pred),'BalancedAccuracy':balanced_accuracy_score(y[test_idx],pred),'Params':search.best_params_,'ConfusionMatrix':confusion_matrix(y[test_idx],pred).tolist()})
    fitted[name]=search
    print(name,results[-1])
metrics=pd.DataFrame(results);display(metrics[['Protocol','Model','CV','Test','BalancedAccuracy']])
(OUT/'model_results.json').write_text(json.dumps(results,indent=2,default=lambda x:float(x)))


Logistic regression {'Protocol': 'Train-only scaling', 'Model': 'Logistic regression', 'CV': np.float64(0.8464285714285713), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'model__C': 0.01, 'model__solver': 'lbfgs'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


SVM {'Protocol': 'Train-only scaling', 'Model': 'SVM', 'CV': np.float64(0.8214285714285714), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'model__C': np.float64(0.03162277660168379), 'model__gamma': np.float64(0.001), 'model__kernel': 'linear'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


Decision tree {'Protocol': 'Train-only scaling', 'Model': 'Decision tree', 'CV': np.float64(0.8767857142857143), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'model__criterion': 'gini', 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 10, 'model__splitter': 'best'}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


KNN {'Protocol': 'Train-only scaling', 'Model': 'KNN', 'CV': np.float64(0.8482142857142858), 'Test': 0.8333333333333334, 'BalancedAccuracy': 0.75, 'Params': {'model__algorithm': 'auto', 'model__n_neighbors': 10, 'model__p': 1}, 'ConfusionMatrix': [[3, 3], [0, 12]]}


,Protocol,Model,CV,Test,BalancedAccuracy
0,Course baseline,Logistic regression,0.846429,0.833333,0.75
1,Course baseline,SVM,0.848214,0.833333,0.75
2,Course baseline,Decision tree,0.876786,0.833333,0.75
3,Course baseline,KNN,0.848214,0.833333,0.75
4,Train-only scaling,Logistic regression,0.846429,0.833333,0.75
5,Train-only scaling,SVM,0.821429,0.833333,0.75
6,Train-only scaling,Decision tree,0.876786,0.833333,0.75
7,Train-only scaling,KNN,0.848214,0.833333,0.75


3149

## Comparison and confusion matrix
The holdout contains only 18 launches, so a one-record difference is 5.56 percentage points. The random split is educational and does not establish future-launch performance. Flight number, serial and historical reuse metadata can encode time or hindsight. A deployment study needs as-of-time features and chronological validation.

In [4]:
clean=metrics[metrics.Protocol.eq('Train-only scaling')].set_index('Model')
fig,ax=plt.subplots();clean[['CV','Test']].mul(100).plot.bar(ax=ax,color=['#187E8A','#DD9E39'],rot=0);ax.set(ylabel='Accuracy (%)',ylim=(0,105));ax.legend(['Training CV','Held-out test']);fig.tight_layout();fig.savefig(FIG/'model_accuracy.png',dpi=150);plt.show()
best=clean.CV.idxmax();pred=fitted[best].predict(X.iloc[test_idx])
fig,ax=plt.subplots(figsize=(7,5));ConfusionMatrixDisplay.from_predictions(y[test_idx],pred,display_labels=['Class 0','Class 1'],ax=ax,cmap='Blues',colorbar=False);ax.set_title(best+' - held-out test');fig.tight_layout();fig.savefig(FIG/'confusion_matrix.png',dpi=150);plt.show()
print('Selected by training CV:',best)
print('Majority-class test baseline:',max(np.mean(y[test_idx]),1-np.mean(y[test_idx])))


Selected by training CV: Decision tree
Majority-class test baseline: 0.6666666666666666


/var/folders/q4/kp0z7j1d201f9tr0p0rvcfgr0000gn/T/ipykernel_83342/1984487555.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();clean[['CV','Test']].mul(100).plot.bar(ax=ax,color=['#187E8A','#DD9E39'],rot=0);ax.set(ylabel='Accuracy (%)',ylim=(0,105));ax.legend(['Training CV','Held-out test']);fig.tight_layout();fig.savefig(FIG/'model_accuracy.png',dpi=150);plt.show()
/var/folders/q4/kp0z7j1d201f9tr0p0rvcfgr0000gn/T/ipykernel_83342/1984487555.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(7,5));ConfusionMatrixDisplay.from_predictions(y[test_idx],pred,display_labels=['Class 0','Class 1'],ax=ax,cmap='Blues',colorbar=False);ax.set_title(best+' - held-out test');fig.tight_layout();fig.savefig(FIG/'confusion_matrix.png',dpi=150);plt.show()
